<figure>
  <IMG SRC="https://upload.wikimedia.org/wikipedia/commons/thumb/d/d5/Fachhochschule_Südwestfalen_20xx_logo.svg/320px-Fachhochschule_Südwestfalen_20xx_logo.svg.png" WIDTH=250 ALIGN="right">
</figure>

# Programmierung für KI
### Winterersemester 2025/26
Prof. Dr. Heiner Giefers

**Übung 1: <br>
Schreiben Sie eine Funktion, die als Parameter eine URL entgegennimmt. Rufen Sie anschließend Daten von der übergebenen URL ab und geben Sie den empfangenen Datensatz als String zurück. Orientieren Sie sich am unten stehenden Beispiel.**

Beispiel<br>
```python
import socket
mysock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
mysock.connect(('data.pr4e.org', 80))
cmd = 'GET http://data.pr4e.org/romeo.txt HTTP/1.0\r\n\r\n'.encode()
mysock.send(cmd)
while True:
    data = mysock.recv(512)
    if len(data) < 1:
        break
    print(data.decode(),end='')
mysock.close()
```

In [1]:
import socket
import re

def load(url):
    host_list = re.findall("https?://([^/]+)", url)
    host = host_list[0]
    mysock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    
    mysock.connect((host, 80))

    cmd = f"GET {url} HTTP/1.0\r\n\r\n".encode()
    mysock.send(cmd)

    received_data_list = []
    
    while True:
        data = mysock.recv(512)
        if len(data) < 1:
            break 
        received_data_list.append(data.decode())

    mysock.close()

    full_data = "".join(received_data_list)

    header_end = full_data.index("\r\n\r\n") + 4
    return full_data[header_end:]

In [2]:
# Test Zelle

from nose.tools import assert_true
assert_true("Arise fair sun and kill the envious moon"\
            in load("http://data.pr4e.org/romeo.txt"), "Der Datensatz ist falsch")

ModuleNotFoundError: No module named 'nose'

**Übung 2: <br>
Schreiben Sie die gleiche Funktion mit der `urllib`. Zählen Sie anschließend die Anzahl der empfangenen Zeichen und geben Sie den Ausdruck `Datensatz zu gross` zurück, wenn mehr als 3000 Zeichen gesendet wurden. Ansonsten geben Sie den Datensatz als String zurück.**

In [3]:
import urllib

def load_urllib(url):
    with urllib.request.urlopen(url) as response:
        content_bytes = response.read()
        charset = response.info().get_content_charset() or 'utf-8'
        data_string = content_bytes.decode(charset)
        char_count = len(data_string)

        if char_count > 3000:
           return "Datensatz zu gross"
        else:
           return data_string

In [4]:
# Test Zelle

from nose.tools import assert_equal, assert_true

assert_true('But soft what light through yonder window'
            in load_urllib("http://data.pr4e.org/romeo.txt"), "Der Datensatz ist falsch")
assert_equal("Datensatz zu gross", 
             load_urllib("https://raw.githubusercontent.com/mbh038/UM_PEP/master/code/mbox-short.txt"), 
             "Der Fehler wurde nicht korrekt behandelt")

ModuleNotFoundError: No module named 'nose'

**Übung 3: <br>
Schreiben Sie eine Funktion, die als Parameter eine URL entgegennimmt. Rufen Sie anschließend Daten von der übergebenen URL ab und zählen Sie die Anzahl der Paragraph-Tags (`<p>`).**

**Hinweis:** Orientieren Sie sich am unten stehenden Beispiel, in dem die Anker-Tags (`<a>`) geparst werden.

```python
import urllib.request, urllib.parse, urllib.error
import ssl
from bs4 import BeautifulSoup

# Ignore SSL certificate errors
ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE
html = urllib.request.urlopen("https://de.wikipedia.org", context=ctx).read()
soup = BeautifulSoup(html, 'html.parser')

# Retrieve all of the anchor tags
tags = soup('a')
for tag in tags:
    print(tag.get('href', None))
```

In [5]:
import urllib.request, urllib.parse, urllib.error
from bs4 import BeautifulSoup
import ssl

def count_para(url):
    # Ignore SSL certificate errors
    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE
    html = urllib.request.urlopen(url, context=ctx).read()
    soup = BeautifulSoup(html, 'html.parser')
    
    # Retrieve all of the anchor tags
    tags = soup('p')
    return len(tags)

In [6]:
# Test Zelle
from nose.tools import assert_equal

assert_equal(2, count_para("https://example.com"), "Die Anzahl der Tags stimmt nicht.")

ModuleNotFoundError: No module named 'nose'